In [ ]:
%pip install transformers
%pip install torch
%pip install tqdm
%pip install accelerate

In [ ]:
from transformers import AutoModel, AutoTokenizer

model_id = "Qwen/Qwen2-1.5B"
model = AutoModel.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

model.save_pretrained("./model")
tokenizer.save_pretrained("./model")

In [ ]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Model
from azure.identity import DefaultAzureCredential

subscription_id = "4e80bb93-03b8-4aaa-8394-7c52710f434a"
resource_group = "ces3-posttraining"
workspace_name = "testwest"

ml_client = MLClient(
    DefaultAzureCredential(),
    subscription_id,
    resource_group,
    workspace_name,
)

model = Model(
    name="hf-qwen-custom-model",
    path="./model",
    type="custom_model",
)

model = ml_client.models.create_or_update(model)


In [ ]:

from azure.ai.ml.entities import ManagedOnlineEndpoint
import datetime

endpoint_name = f"hf-qwen-endpoint-{datetime.datetime.now().strftime('%Y%m%d%H%M%S')}"

endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    auth_mode="key",  # or "aad_token"
    description="qwen2 inference endpoint",
)

ml_client.online_endpoints.begin_create_or_update(endpoint).result()

In [ ]:
from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment, Model, CodeConfiguration, ProbeSettings
print(f"endpoint: {endpoint_name}")
for e in ml_client.environments.list(name="hf-env"):
  print(e.name, e.version)
  break

print(f"env:{e.name}:{e.version}")
deployment = ManagedOnlineDeployment(
    name="green",
    endpoint_name=endpoint_name,
    model=model,                 # or "hf-model:1"
    environment=f"{e.name}:{e.version}",  # or "hf-env@latest"
    code_configuration=CodeConfiguration(code="./src", scoring_script="scoreqwen.py"),
    instance_type="Standard_NC4as_T4_v3",
    # instance_type="Standard_NC40ads_H100_v5",
    instance_count=1,    
    environment_variables={
        "AZUREML_HTTP_PORT": "5001"
    },
    liveness_probe=ProbeSettings(
        initial_delay=300,
        timeout=10,
        period=10,
        failure_threshold=6,
    ),
    readiness_probe=ProbeSettings(
        initial_delay=300,
        timeout=10,
        period=10,
        failure_threshold=6,
    ),


)

ml_client.online_deployments.begin_create_or_update(deployment).result()

In [ ]:

# Fetch endpoint credentials
creds = ml_client.online_endpoints.get_keys(
    name=endpoint_name
)

# API keys (for auth_mode: key)
primary_key = creds.primary_key
secondary_key = creds.secondary_key

print("Primary key:", primary_key)
print("Secondary key:", secondary_key)
# 6) Route traffic to the deployment
endpoint = ml_client.online_endpoints.get(name=endpoint_name)
endpoint.traffic = {"green": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

In [8]:
import json
import torch
import os
import logging
from transformers import AutoTokenizer, OPTForCausalLM, AutoModelForCausalLM
MODEL_NAME = os.environ.get("HF_MODEL_NAME", "Qwen/Qwen2-1.5B-Instruct")
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)


# Globals loaded once
tokenizer = None
model = None
device = None


def init():

    global tokenizer, model, device

    base = os.environ["AZUREML_MODEL_DIR"]              # e.g. /var/azureml-app/azureml-models/.../1
    model_dir = os.path.join(base, "model")             # <- because your files are in .../1/model/

    logger.info(f"AZUREML_MODEL_DIR={base}")
    logger.info(f"Using model_dir={model_dir}")
    logger.info(f"Files in model_dir: {os.listdir(model_dir)}")

    # Pick device
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Load tokenizer + model
    # Qwen2 model card recommends transformers>=4.37.0; it also uses apply_chat_template. [1](https://huggingface.co/Qwen/Qwen2-1.5B-Instruct)[2](https://huggingface.co/docs/transformers/model_doc/qwen2)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype="auto",
        device_map="auto" if device == "cuda" else None,
    )

    if device != "cuda":
        model.to(device)

    model.eval()

# def init():
#    global model, tokenizer
#    model_dir = "./model"  # Azure mounts model here automatically
#    tokenizer = AutoTokenizer.from_pretrained(model_dir)
#    model = OPTForCausalLM.from_pretrained(model_dir)
#    model.eval()


def _build_messages(payload: dict):
    # Accept either a raw prompt or full chat messages
    if "messages" in payload and isinstance(payload["messages"], list):
        return payload["messages"]

    prompt = payload.get("input") or payload.get("text") or payload.get("prompt") or ""
    system = payload.get("system", "You are a helpful assistant.")
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt},
    ]


def run(raw_data):
    logger.info(f"raw_data={raw_data}")
    try:
        payload = json.loads(raw_data) if isinstance(raw_data, (str, bytes, bytearray)) else raw_data
        messages = _build_messages(payload)
        logger.info(f"messages={messages}")

        # Qwen examples format chat with apply_chat_template(add_generation_prompt=True). [1](https://huggingface.co/Qwen/Qwen2-1.5B-Instruct)[3](https://qwen.readthedocs.io/en/v2.0/inference/chat.html)
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer([text], return_tensors="pt")
        inputs = {k: v.to(model.device) for k, v in inputs.items()}


        # Generation params (override via request if you want)
        max_new_tokens = int(payload.get("max_new_tokens", 256))
        temperature = float(payload.get("temperature", 0.7))
        top_p = float(payload.get("top_p", 0.90))
        top_k = float(payload.get("top_k", 50))
        do_sample = bool(payload.get("do_sample", False))

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=do_sample,
                temperature=temperature,
                top_p=top_p,
            )

        # Remove prompt tokens from output (as in model card). [1](https://huggingface.co/Qwen/Qwen2-1.5B-Instruct)[3](https://qwen.readthedocs.io/en/v2.0/inference/chat.html)
        gen_only = generated_ids[:, inputs["input_ids"].shape[1]:]
        response_text = tokenizer.batch_decode(gen_only, skip_special_tokens=True)[0]
        logger.info(f"generated_text={response_text}")
        return {
            "generated_text": response_text,
            "model": MODEL_NAME,
            "device": str(model.device),
        }

    except Exception as e:
        # Keep errors JSON-serializable for AzureML inference server
        return {"error": str(e)}

In [ ]:
import json
import torch
import os
import logging
from transformers import AutoTokenizer, OPTForCausalLM, AutoModelForCausalLM
MODEL_NAME = os.environ.get("HF_MODEL_NAME", "Qwen/Qwen2-1.5B-Instruct")
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)


# Globals loaded once
tokenizer = None
model = None
device = None


def init():

    global tokenizer, model, device

    base = os.environ.get("AZUREML_MODEL_DIR",".")              # e.g. /var/azureml-app/azureml-models/.../1
    model_dir = os.path.join(base, "model")             # <- because your files are in .../1/model/

    logger.info(f"AZUREML_MODEL_DIR={base}")
    logger.info(f"Using model_dir={model_dir}")
    logger.info(f"Files in model_dir: {os.listdir(model_dir)}")

    # Pick device
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Load tokenizer + model
    # Qwen2 model card recommends transformers>=4.37.0; it also uses apply_chat_template. [1](https://huggingface.co/Qwen/Qwen2-1.5B-Instruct)[2](https://huggingface.co/docs/transformers/model_doc/qwen2)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype="auto",
        #load_in_4bit=True,
        device_map="auto" if device == "cuda" else None,
        #device_map="cuda",
    )

    if device != "cuda":
        model.to(device)

    model.eval()

# def init():
#    global model, tokenizer
#    model_dir = "./model"  # Azure mounts model here automatically
#    tokenizer = AutoTokenizer.from_pretrained(model_dir)
#    model = OPTForCausalLM.from_pretrained(model_dir)
#    model.eval()


def _build_messages(payload: dict):
    # Accept either a raw prompt or full chat messages
    if "messages" in payload and isinstance(payload["messages"], list):
        return payload["messages"]

    prompt = payload.get("input") or payload.get("text") or payload.get("prompt") or ""
    system = payload.get("system", "You are a helpful assistant.")
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt},
    ]


def run(raw_data):
    logger.info(f"raw_data={raw_data}")
    try:
        payload = json.loads(raw_data) if isinstance(raw_data, (str, bytes, bytearray)) else raw_data
        messages = _build_messages(payload)
        logger.info(f"messages={messages}")

        # Qwen examples format chat with apply_chat_template(add_generation_prompt=True). [1](https://huggingface.co/Qwen/Qwen2-1.5B-Instruct)[3](https://qwen.readthedocs.io/en/v2.0/inference/chat.html)
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer([text], return_tensors="pt")
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        # Generation params (override via request if you want)
        max_new_tokens = int(payload.get("max_new_tokens", 256))
        temperature = float(payload.get("temperature", 0.7))
        top_p = float(payload.get("top_p", 0.95))
        do_sample = bool(payload.get("do_sample", temperature > 0))

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=do_sample,
                temperature=temperature,
                top_p=top_p,
            )

        # Remove prompt tokens from output (as in model card). [1](https://huggingface.co/Qwen/Qwen2-1.5B-Instruct)[3](https://qwen.readthedocs.io/en/v2.0/inference/chat.html)
        gen_only = generated_ids[:, inputs["input_ids"].shape[1]:]
        response_text = tokenizer.batch_decode(gen_only, skip_special_tokens=True)[0]
        logger.info(f"generated_text={response_text}")
        return {
            "generated_text": response_text,
            "model": MODEL_NAME,
            "device": str(model.device),
        }

    except Exception as e:
        # Keep errors JSON-serializable for AzureML inference server
        return {"error": str(e)}

init()
input_data = "{\"messages\":[{\"role\":\"system\",\"content\":\"You are a helpful assistant.\"},{\"role\":\"user\",\"content\":\"I am going to Paris, what should I see?\"}]}"
print(f"input_data: {input_data} ")
result = run(input_data)
print(f"result: {result}")
